# Proyek Klasifikasi Gambar: Klasifikasi Sampah
- **Nama:** Robil
- **Email:** robilputra19@gmail.com
- **ID Dicoding:** robilputra19

## Import Semua Packages/Library yang Digunakan

In [3]:
import os
import shutil
import zipfile
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.model_selection import train_test_split

print('TensorFlow version:', tf.__version__)
print('Num GPUs Available:', len(tf.config.list_physical_devices('GPU')))

KeyboardInterrupt: 

In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU siap dipakai 🚀")
    except RuntimeError as e:
        print(e)
else:
    print("GPU tidak terdeteksi ❌")

## Data Preparation

### Data Loading

In [ ]:
import shutil
import os

extract_path = '/content/drive/MyDrive/dataset/dataset-sampah'

# hapus folder lama kalau ada
if os.path.exists(extract_path):
    shutil.rmtree(extract_path)
    print("Folder lama dihapus ✅")

In [ ]:
import zipfile

zip_path = '/content/drive/MyDrive/dataset/archive.zip'
extract_path = '/content/drive/MyDrive/dataset/dataset-sampah'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print('Dataset berhasil diekstrak ke Drive 🚀')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Eksplorasi struktur dataset
BASE_DIR = '/content/drive/MyDrive/dataset/dataset-sampah/dataset-sampah'

# Cek struktur folder
for root, dirs, files in os.walk(BASE_DIR):
    level = root.replace(BASE_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level == 1:
        print(f'{indent}  -> {len(files)} gambar')

In [ ]:
# BASE_DIR sudah benar — kelas langsung ada di dalam 'dataset-sampah/'
# Struktur: dataset-sampah/botol plastik/, dataset-sampah/kaca/, dst.

CLASS_NAMES = sorted([
    c for c in os.listdir(BASE_DIR)
    if os.path.isdir(os.path.join(BASE_DIR, c))
])

print('Kelas yang ditemukan:', CLASS_NAMES)
print('Jumlah kelas         :', len(CLASS_NAMES))

# Hitung total gambar per kelas
total_images = 0
print()
for cls in CLASS_NAMES:
    cls_path = os.path.join(BASE_DIR, cls)
    images = [
        f for f in os.listdir(cls_path)
        if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))
    ]
    total_images += len(images)
    print(f'  {cls:<20}: {len(images)} gambar')

print(f'\nTotal gambar: {total_images}')

In [ ]:
# Visualisasi sampel gambar dari setiap kelas
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, cls in enumerate(CLASS_NAMES):
    cls_path = os.path.join(BASE_DIR, cls)
    img_files = os.listdir(cls_path)
    sample_img = os.path.join(cls_path, random.choice(img_files))
    img = mpimg.imread(sample_img)
    axes[i].imshow(img)
    axes[i].set_title(cls, fontsize=12)
    axes[i].axis('off')

# Sembunyikan axis yang tidak dipakai
for j in range(len(CLASS_NAMES), len(axes)):
    axes[j].axis('off')

plt.suptitle('Sampel Gambar per Kelas', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### Data Preprocessing

#### Split Dataset

In [ ]:
# ============================================================
# Split dataset: 70% Train, 15% Validation, 15% Test
# ============================================================

SPLIT_DIR = 'split_dataset'
TRAIN_DIR = os.path.join(SPLIT_DIR, 'train')
VAL_DIR   = os.path.join(SPLIT_DIR, 'validation')
TEST_DIR  = os.path.join(SPLIT_DIR, 'test')

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

random.seed(42)

for split in ['train', 'validation', 'test']:
    for cls in CLASS_NAMES:
        os.makedirs(os.path.join(SPLIT_DIR, split, cls), exist_ok=True)

for cls in CLASS_NAMES:
    cls_path = os.path.join(BASE_DIR, cls)
    images = [f for f in os.listdir(cls_path)
              if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))]
    random.shuffle(images)

    n = len(images)
    n_train = int(n * TRAIN_RATIO)
    n_val   = int(n * VAL_RATIO)

    train_imgs = images[:n_train]
    val_imgs   = images[n_train:n_train + n_val]
    test_imgs  = images[n_train + n_val:]

    for img in train_imgs:
        shutil.copy(os.path.join(cls_path, img), os.path.join(TRAIN_DIR, cls, img))
    for img in val_imgs:
        shutil.copy(os.path.join(cls_path, img), os.path.join(VAL_DIR, cls, img))
    for img in test_imgs:
        shutil.copy(os.path.join(cls_path, img), os.path.join(TEST_DIR, cls, img))

    print(f'{cls}: Train={len(train_imgs)}, Val={len(val_imgs)}, Test={len(test_imgs)}')

print('\nDataset berhasil di-split!')

In [ ]:
# ============================================================
# Konfigurasi ImageDataGenerator — 224x224 untuk Transfer Learning
# ============================================================

IMG_SIZE   = (224, 224)
BATCH_SIZE = 32

# Augmentasi untuk data training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

# Hanya rescale untuk validation dan test
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    seed=42
)

val_generator = val_test_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print('Label Index Mapping:')
print(train_generator.class_indices)


## Modelling

In [ ]:
# ============================================================
# Model: Transfer Learning dengan MobileNetV2 (Sequential)
# MobileNetV2 berisi Conv2D + DepthwiseConv2D + Pooling layers
# ============================================================

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D

NUM_CLASSES = len(CLASS_NAMES)

# Load MobileNetV2 pretrained ImageNet — tanpa top layer
base_model = MobileNetV2(
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3),
    include_top=False,
    weights='imagenet'
)

# === FASE 1: Freeze semua layer base model ===
base_model.trainable = False
print(f'Total layer MobileNetV2 : {len(base_model.layers)}')
print(f'Trainable params (fase 1): {sum([tf.size(w).numpy() for w in base_model.trainable_weights])}')

model = Sequential([
    base_model,                                      # Conv2D + Pooling ada di sini
    GlobalAveragePooling2D(),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(NUM_CLASSES, activation='softmax')
], name='WasteClassifier_MobileNetV2')

model.summary()


In [ ]:
# ============================================================
# FASE 1 — Compile & Training: Hanya top layer (base frozen)
# ============================================================

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_phase1 = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]


In [ ]:
# ============================================================
# FASE 1 Training — Epoch awal, base model frozen
# ============================================================

print('=' * 50)
print('FASE 1: Training top layers (base model frozen)')
print('=' * 50)

history_phase1 = model.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator,
    callbacks=callbacks_phase1,
    verbose=1
)

print(f'\nFase 1 selesai! Val Accuracy: {max(history_phase1.history["val_accuracy"])*100:.2f}%')

# ============================================================
# FASE 2 — Fine-tune: Unfreeze 50 layer terakhir MobileNetV2
# ============================================================

print('\n' + '=' * 50)
print('FASE 2: Fine-tuning (unfreeze top 50 layers base model)')
print('=' * 50)

base_model.trainable = True
# Freeze semua kecuali 50 layer terakhir
for layer in base_model.layers[:-50]:
    layer.trainable = False

trainable_count = sum([1 for l in base_model.layers if l.trainable])
print(f'Layer yang di-unfreeze: {trainable_count} dari {len(base_model.layers)}')

# Compile ulang dengan learning rate lebih kecil
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_phase2 = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=4,
        min_lr=1e-7,
        verbose=1
    )
]

history_phase2 = model.fit(
    train_generator,
    epochs=30,
    validation_data=val_generator,
    callbacks=callbacks_phase2,
    verbose=1
)

print(f'\nFase 2 selesai! Val Accuracy: {max(history_phase2.history["val_accuracy"])*100:.2f}%')

# Gabungkan history kedua fase
import collections
history_combined = collections.defaultdict(list)
for key in history_phase1.history:
    history_combined[key] = history_phase1.history[key] + history_phase2.history[key]


## Evaluasi dan Visualisasi

In [ ]:
# ============================================================
# Evaluasi Model pada Test Set
# ============================================================

print('=== Evaluasi pada Training Set ===')
train_loss, train_acc = model.evaluate(train_generator, verbose=1)
print(f'Train Loss    : {train_loss:.4f}')
print(f'Train Accuracy: {train_acc:.4f} ({train_acc*100:.2f}%)')

print('\n=== Evaluasi pada Validation Set ===')
val_loss, val_acc = model.evaluate(val_generator, verbose=1)
print(f'Val Loss    : {val_loss:.4f}')
print(f'Val Accuracy: {val_acc:.4f} ({val_acc*100:.2f}%)')

print('\n=== Evaluasi pada Test Set ===')
test_loss, test_acc = model.evaluate(test_generator, verbose=1)
print(f'Test Loss    : {test_loss:.4f}')
print(f'Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)')

In [ ]:
# ============================================================
# Plot Akurasi dan Loss (Gabungan Fase 1 + Fase 2)
# ============================================================

# Tentukan batas fase
n_fase1 = len(history_phase1.history['accuracy'])
epochs_total = len(history_combined['accuracy'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot Accuracy
axes[0].plot(history_combined['accuracy'],     label='Train Accuracy', color='royalblue',  linewidth=2)
axes[0].plot(history_combined['val_accuracy'], label='Val Accuracy',   color='darkorange', linewidth=2)
axes[0].axvline(x=n_fase1, color='gray', linestyle='--', linewidth=1.5, label=f'Fine-tune mulai (epoch {n_fase1})')
axes[0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.6)
axes[0].set_ylim([0, 1])

# Plot Loss
axes[1].plot(history_combined['loss'],     label='Train Loss', color='royalblue',  linewidth=2)
axes[1].plot(history_combined['val_loss'], label='Val Loss',   color='darkorange', linewidth=2)
axes[1].axvline(x=n_fase1, color='gray', linestyle='--', linewidth=1.5, label=f'Fine-tune mulai (epoch {n_fase1})')
axes[1].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.suptitle('Training History — Waste Classification (MobileNetV2 + Fine-Tuning)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot tersimpan sebagai training_history.png')


## Konversi Model

In [ ]:
# ============================================================
# 1. Simpan dalam format SavedModel
# ============================================================

SAVED_MODEL_DIR = 'saved_model'
model.save(SAVED_MODEL_DIR)
print(f'SavedModel tersimpan di: {SAVED_MODEL_DIR}')
print('Isi folder saved_model:')
for item in os.listdir(SAVED_MODEL_DIR):
    print(' ', item)

In [ ]:
# ============================================================
# 2. Simpan dalam format TF-Lite
# ============================================================

os.makedirs('tflite', exist_ok=True)

# Konversi ke TFLite
converter = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL_DIR)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

# Simpan model
tflite_path = os.path.join('tflite', 'model.tflite')
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

# Simpan label.txt
label_path = os.path.join('tflite', 'label.txt')
with open(label_path, 'w') as f:
    for cls in CLASS_NAMES:
        f.write(cls + '\n')

print(f'TFLite model tersimpan di: {tflite_path}')
print(f'Label file tersimpan di  : {label_path}')
print(f'Ukuran TFLite model: {os.path.getsize(tflite_path) / 1024 / 1024:.2f} MB')

In [ ]:
# ============================================================
# 3. Simpan dalam format TensorFlow.js (TFJS)
# ============================================================

!pip install tensorflowjs -q

import tensorflowjs as tfjs

TFJS_DIR = 'tfjs_model'
os.makedirs(TFJS_DIR, exist_ok=True)

tfjs.converters.save_keras_model(model, TFJS_DIR)

print(f'TFJS model tersimpan di: {TFJS_DIR}')
print('Isi folder tfjs_model:')
for item in os.listdir(TFJS_DIR):
    print(' ', item)

In [ ]:
# ============================================================
# Verifikasi semua format model
# ============================================================

print('=== Verifikasi Model SavedModel ===')
loaded_model = tf.saved_model.load(SAVED_MODEL_DIR)
print('SavedModel berhasil dimuat!')

print('\n=== Verifikasi TFLite ===')
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()
input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()
print(f'Input shape : {input_details[0]["shape"]}')
print(f'Output shape: {output_details[0]["shape"]}')
print('TFLite model berhasil diverifikasi!')

print('\n=== Ringkasan Konversi ===')
print(f'  SavedModel : {SAVED_MODEL_DIR}/')
print(f'  TF-Lite    : {tflite_path}')
print(f'  TFJS       : {TFJS_DIR}/')
print('Semua format model berhasil disimpan!')

## Inference (Optional)

In [ ]:
# ============================================================
# Inference: Prediksi pada beberapa gambar dari test set
# ============================================================

from tensorflow.keras.preprocessing import image as keras_image

label_map = {v: k for k, v in train_generator.class_indices.items()}

def predict_image(img_path, model, label_map, img_size=(224, 224)):
    img = keras_image.load_img(img_path, target_size=img_size)
    img_array = keras_image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    predictions = model.predict(img_array, verbose=0)
    pred_class  = label_map[np.argmax(predictions[0])]
    confidence  = np.max(predictions[0]) * 100
    return pred_class, confidence, predictions[0]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, cls in enumerate(CLASS_NAMES):
    cls_test_path = os.path.join(TEST_DIR, cls)
    img_files = [f for f in os.listdir(cls_test_path)
                 if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))]
    if not img_files:
        continue
    sample = os.path.join(cls_test_path, random.choice(img_files))
    pred_class, confidence, _ = predict_image(sample, model, label_map)
    img = mpimg.imread(sample)
    axes[i].imshow(img)
    color = 'green' if pred_class == cls else 'red'
    axes[i].set_title(f'True: {cls}\nPred: {pred_class} ({confidence:.1f}%)',
                      fontsize=9, color=color)
    axes[i].axis('off')

for j in range(len(CLASS_NAMES), len(axes)):
    axes[j].axis('off')

plt.suptitle('Hasil Prediksi pada Test Set (Hijau=Benar, Merah=Salah)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Buat file requirements.txt
# ============================================================

requirements = """tensorflow>=2.12.0
numpy>=1.23.0
matplotlib>=3.6.0
scikit-learn>=1.1.0
tensorflowjs>=4.0.0
Pillow>=9.0.0
kaggle>=1.5.12
"""

with open('requirements.txt', 'w') as f:
    f.write(requirements)

print('requirements.txt berhasil dibuat!')
print(requirements)

In [ ]:
# ============================================================
# ZIP semua file submission
# ============================================================

import zipfile

submission_zip = 'submission.zip'

def add_dir_to_zip(zf, src_dir, arcname=None):
    for root, dirs, files in os.walk(src_dir):
        for file in files:
            file_path = os.path.join(root, file)
            arc = os.path.join(arcname or src_dir,
                               os.path.relpath(file_path, src_dir))
            zf.write(file_path, arc)

with zipfile.ZipFile(submission_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    # SavedModel
    add_dir_to_zip(zf, 'saved_model', 'submission/saved_model')
    # TFJS
    add_dir_to_zip(zf, 'tfjs_model', 'submission/tfjs_model')
    # TFLite
    add_dir_to_zip(zf, 'tflite', 'submission/tflite')
    # Notebook
    if os.path.exists('notebook.ipynb'):
        zf.write('notebook.ipynb', 'submission/notebook.ipynb')
    # Requirements
    zf.write('requirements.txt', 'submission/requirements.txt')

print(f'Submission zip tersimpan: {submission_zip}')
print(f'Ukuran file: {os.path.getsize(submission_zip) / 1024 / 1024:.2f} MB')